# Diagnostico del entrenamiento

Compara variantes del entrenamiento de main.py con un mismo K y una misma semilla. Cada conjunto de variantes es un archivo de `jobs/diag_variants/` (una linea por variante: su nombre y las opciones de main.py que cambia) y se lanza con `make run-diag DIAG_SET=<conjunto>`, un trabajo de GPU por conjunto.

| Conjunto | Pregunta | Resultado |
|---|---|---|
| `caida` | ¿FedAvg se cae cuando ε llega a su piso por el buffer chico o por la recompensa sin escalar? | No: ninguna variante deja de caerse |
| `aprendizaje` | ¿El agente no encuentra la regla de umbral por la red chica, por dejar de explorar o por γ = 0.99? Prueba la red y el ε final de la Tabla III del paper, y γ = 0.9 | No le gana a Greedy ninguna. γ = 0.9 casi quita la caida (−8 % contra −34 %) y termina en ~4700; la red y el ε del paper apenas ayudan |
| `cola` | ¿Aprende a transmitir cuando la cola se llena si el costo de callarse se cobra en el slot que lo causa (`--reward-end-of-slot-queue`)? | No: sola no cambia nada (−33 %); con γ = 0.9 queda como γ = 0.9 sin ella (~4850, la mejor, pero ~800 por debajo de Greedy) |
| `recompensa` | ¿Aprende la regla si cada agente recibe su propia recompensa (`--local-reward`: su cola, su confiabilidad y el CUE de su canal) en vez de la compartida? El CSV guarda la compartida, asi que se compara igual | Es la mayor mejora: las cuatro superan a DTE y a toda variante con la recompensa compartida, y a mitad del entrenamiento a Greedy (5800–6050). Al terminar la exploracion olvidan una parte; la mejor (con γ = 0.9 y la cola al final del slot) termina en ~5590, casi igual a Greedy |
| `gamma` | ¿Hay un γ mejor cerca de 0.9 (0.95, 0.85, 0.8, 0.7)? | No: 0.9 sigue siendo el mejor (~4700) y 0.85 queda igual (~4650); 0.95 se cae al final (~3330) y 0.8 y 0.7 quedan peor (~4300) |
| `estabilidad` | Sobre la mejor de `recompensa`, ¿una tasa de aprendizaje mas baja (`--lr`), mas exploracion (`--epsilon-end 0.1`) o un buffer mas grande evitan que olvide al terminar la exploracion? | No: ninguna evita el olvido. Las cuatro terminan entre ~5130 y ~5390, por debajo de la mejor sin cambios (~5590); `--lr 3e-4` llega al pico mas alto (~6230) pero olvida igual (−14 %). La configuracion queda como esta |

**La regla de umbral.** `agents/threshold_policy.py`, sin entrenar y solo con informacion local: un enlace fuerte (ganancia media de su canal propio de −91 dB o mas, ~25 m) transmite a potencia maxima; uno debil se queda callado, para no interferir al CUE de su canal, hasta que su cola llega al 90 % de Q_max. Con K = 10 le gana a Greedy por ~25 %, asi que hay algo que aprender: es el blanco minimo de un agente que funciona.

**Que buscar.** Una variante que (1) no se caiga cuando ε llega al piso, (2) termine por encima de su arranque casi al azar y (3) le gane a Greedy y a DTE con el mismo K. Mejor aun si se acerca a la regla de umbral. Esa seria la configuracion para volver a correr FedAvg, DTE y gossip.

**Referencias.** Greedy, DTE y la regla de umbral con el mismo K y la misma semilla, de `runs/k<K>/` (`make run-umbral` corre la regla en Hypatia). DTE corrio 4000 episodios y estas variantes 2000, asi que su linea es una referencia, no una comparacion exacta. Greedy y la regla no aprenden, asi que su valor no depende de los episodios.

**Datos.** En Hypatia, desde la raiz del repo: `make run-diag DIAG_SET=<conjunto>`. Cuando termine, en la maquina local: `make pull-runs`. Para ver otros conjuntos u otro K, cambia `DIAG_SETS` o `DIAG_K` en la celda siguiente.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")  # v2v/, para importar analysis.py y plots.py
from analysis import final_values, load_metadata, load_runs, per_episode_means
from plots import LABELS, METHODS, REFERENCE, use_thesis_style

RUNS_ROOT = Path(os.environ.get("V2V_RUNS_ROOT", "../runs"))
FIG_DIR = Path(os.environ.get("V2V_FIG_DIR", "../runs/figures"))
FIG_DIR.mkdir(parents=True, exist_ok=True)
VARIANTS_DIR = Path("../jobs/diag_variants")

# Archivos de jobs/diag_variants/, sin el .txt; una variante que esta en varios (como base) aparece una vez.
DIAG_SETS = os.environ.get("V2V_DIAG_SETS", "recompensa,gamma,estabilidad").split(",")
DIAG_K = int(os.environ.get("V2V_DIAG_K", 10))  # el K y la semilla con que se lanzo run-diag
DIAG_SEED = int(os.environ.get("V2V_DIAG_SEED", 0))
REFERENCE_METHODS = ["greedy", "dte", "umbral"]
MUST_BEAT = ["greedy", "dte"]  # criterio (3); la regla de umbral se reporta, pero no se exige
COLORS = [
    "#2a78d6", "#d6457b", "#16917d", "#b5651d", "#52514e", "#0f5c8a", "#9c2f5a", "#5b8c1f", "#6d4bb8",
    "#e8590c", "#3b5bdb", "#2b8a3e", "#c92a2a", "#868e96", "#f08c00", "#1098ad", "#5f3dc4",
]
WINDOW = 50  # episodios de la media movil
FINAL_FRACTION = 0.1  # tramo final y tramo inicial: el 10 % de los episodios
EPS_FLOOR_FRACTION = 0.6  # main.epsilon_schedule: epsilon llega a su piso al 60 % del entrenamiento
COLLAPSE_TOLERANCE = 0.05  # "se cae": el final queda mas de un 5 % por debajo de su mejor tramo
use_thesis_style()

## Carga

Las variantes de los conjuntos, con las opciones que cambian, cuantos episodios tienen y cuanto tardaron. Las referencias aparecen abajo; si falta alguna, las secciones que la usan lo dicen.

In [ ]:
def read_variants(path: Path) -> dict[str, dict]:
    """Lee jobs/diag_variants/<conjunto>.txt igual que jobs/diag.sbatch: {nombre: opciones y metodo}."""
    variants = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip() or line.lstrip().startswith("#"):
            continue
        name, _, flags = line.strip().partition(" ")
        tokens = flags.split()
        method = "fedavg"
        for option, value in zip(tokens, tokens[1:]):
            if option == "--aggregation":
                method = {"none": "dte", "gossip": "gossip"}.get(value, "fedavg")
        variants[name] = {"flags": flags.strip(), "method": method}
    return variants


SETS = {name: read_variants(VARIANTS_DIR / f"{name}.txt") for name in DIAG_SETS}
VARIANTS = {}
for set_variants in SETS.values():
    for name, variant in set_variants.items():
        VARIANTS.setdefault(name, {**variant, "label": name, "color": COLORS[len(VARIANTS) % len(COLORS)]})

frames, meta_rows = [], []
for name, variant in VARIANTS.items():
    root = RUNS_ROOT / "diag" / name
    method = variant["method"]
    if not (root / f"k{DIAG_K}" / f"{method}_seed{DIAG_SEED}.csv").exists():
        print(f"Falta la variante {name} (se esperaba {root}/k{DIAG_K}/{method}_seed{DIAG_SEED}.csv).")
        continue
    run = per_episode_means(load_runs(root, methods=[method]))
    frames.append(run[(run["k"] == DIAG_K) & (run["seed"] == DIAG_SEED)].assign(variant=name))
    meta_rows.append(load_metadata(root, methods=[method]).assign(variant=name))
if not frames:
    raise FileNotFoundError(
        f"No hay ninguna variante de {', '.join(DIAG_SETS)} en {RUNS_ROOT / 'diag'}: "
        "corre `make run-diag DIAG_SET=<conjunto>` y `make pull-runs`."
    )
diag = pd.concat(frames, ignore_index=True)
variants = [v for v in VARIANTS if v in set(diag["variant"])]
episodes = int(diag["episode"].max()) + 1
eps_floor_episode = EPS_FLOOR_FRACTION * episodes

meta = pd.concat(meta_rows, ignore_index=True).set_index("variant")
coverage = pd.DataFrame(
    {
        "Opciones": {v: VARIANTS[v]["flags"] or "(ninguna)" for v in variants},
        "Metodo": {v: METHODS[VARIANTS[v]["method"]]["label"] for v in variants},
        "Episodios": diag.groupby("variant")["episode"].count(),
        "Horas": meta["wall_time_s"] / 3600,
    }
).reindex(variants)

references = {}
try:
    sweep = per_episode_means(load_runs(RUNS_ROOT, methods=REFERENCE_METHODS))
    sweep = sweep[(sweep["k"] == DIAG_K) & (sweep["seed"] == DIAG_SEED)]
    for method, run in sweep.groupby("method"):
        references[method] = final_values(run, FINAL_FRACTION).iloc[0]
except FileNotFoundError:
    pass
references = {m: references[m] for m in REFERENCE_METHODS if m in references}
missing_refs = [m for m in REFERENCE_METHODS if m not in references]
if missing_refs:
    print(f"Sin referencia para: {', '.join(missing_refs)} (K = {DIAG_K}, semilla {DIAG_SEED}, en {RUNS_ROOT}/k{DIAG_K}/).")
coverage.round(2)

## Curvas de aprendizaje

Recompensa por episodio (media movil de `WINDOW` episodios), un panel por conjunto. La linea vertical marca el episodio en que ε llega a su piso; en el barrido, la caida empezaba poco antes de ese punto. Lo que hay que ver: cual variante sigue subiendo, o al menos se mantiene, despues de la linea, y si se acerca a la regla de umbral.

In [ ]:
def moving(run: pd.DataFrame, metric: str) -> pd.Series:
    run = run.sort_values("episode")
    return run.set_index("episode")[metric].rolling(WINDOW, min_periods=WINDOW).mean()


fig, axes = plt.subplots(len(SETS), 1, figsize=(7.5, 3.6 * len(SETS)), squeeze=False)
for ax, (set_name, set_variants) in zip(axes[:, 0], SETS.items()):
    for name in [v for v in set_variants if v in variants]:
        ax.plot(moving(diag[diag["variant"] == name], "reward"), color=VARIANTS[name]["color"], label=name)
    for method, ref in references.items():
        style = METHODS[method]
        ax.axhline(ref["reward"], color=style["color"], linestyle="--", linewidth=1.2, label=style["label"])
    ax.axvline(eps_floor_episode, color=REFERENCE, linestyle=":", linewidth=1)
    ax.annotate("ε en su piso", (eps_floor_episode, 1), xycoords=("data", "axes fraction"), xytext=(4, -12),
                textcoords="offset points", color=REFERENCE, fontsize=8)
    ax.set_title(f"{LABELS['reward']}: conjunto {set_name}, K = {DIAG_K}, semilla {DIAG_SEED}")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=7.5)
axes[-1, 0].set_xlabel("Episodio")
fig.tight_layout()
fig.savefig(FIG_DIR / "diagnostico_curvas.pdf")
plt.show()

## Que cambia en el comportamiento

Tasa CUE, outage y potencia media, un renglon por conjunto. En el barrido y en el conjunto `caida`, la caida de FedAvg venia con menos potencia, mas tasa CUE y mucho mas outage: los pares se callaban para proteger a los CUEs, todos a la vez, y las colas se llenaban. La regla de umbral tambien calla a muchos pares (su outage es alto porque un par callado cuenta como outage), pero mantiene a potencia maxima los enlaces fuertes y vacia la cola de los debiles a tiempo.

In [ ]:
panels = [("cue_rate_mbps", 1), ("outage_rate", 100), ("mean_power_ratio", 1), ("v2v_delay_ms", 1)]
fig, axes = plt.subplots(len(SETS), len(panels), figsize=(12, 2.9 * len(SETS)), squeeze=False)
for row, (set_name, set_variants) in zip(axes, SETS.items()):
    for ax, (metric, factor) in zip(row, panels):
        for name in [v for v in set_variants if v in variants]:
            ax.plot(moving(diag[diag["variant"] == name], metric) * factor, color=VARIANTS[name]["color"], label=name)
        for method, ref in references.items():
            ax.axhline(ref[metric] * factor, color=METHODS[method]["color"], linestyle="--", linewidth=1.1)
        ax.axvline(eps_floor_episode, color=REFERENCE, linestyle=":", linewidth=1)
        ax.set_title(f"{LABELS[metric]} ({set_name})", fontsize=9)
    row[0].legend(fontsize=6.5)
for ax in axes[-1]:
    ax.set_xlabel("Episodio")
fig.tight_layout()
fig.savefig(FIG_DIR / "diagnostico_comportamiento.pdf")
plt.show()

## Tabla

- **Casi al azar**: promedio del primer 10 % de los episodios, cuando ε todavia esta entre 1 y ~0.85.
- **Mejor tramo**: el maximo de la media movil de `WINDOW` episodios.
- **Final**: promedio del ultimo 10 %, como en el resto de la tesis.
- **Caida**: cuanto queda el final por debajo del mejor tramo, en %.

In [ ]:
rows = {}
for name in variants:
    run = diag[diag["variant"] == name].sort_values("episode")
    n = max(1, int(np.ceil(FINAL_FRACTION * len(run))))
    start, end = run.head(n), run.tail(n)
    best = moving(run, "reward").max()
    row = {
        "Casi al azar": start["reward"].mean(),
        "Mejor tramo": best,
        "Final": end["reward"].mean(),
        "Caida (%)": 100 * (end["reward"].mean() - best) / abs(best),
    }
    for method, ref in references.items():
        row[f"Final - {METHODS[method]['label']}"] = end["reward"].mean() - ref["reward"]
    for metric in ("cue_rate_mbps", "outage_rate", "mean_power_ratio", "v2v_delay_ms", "channels_used"):
        row[LABELS[metric]] = end[metric].mean() * (100 if metric == "outage_rate" else 1)
    rows[name] = row
table = pd.DataFrame(rows).T
for method, ref in references.items():
    reference_row = {"Final": ref["reward"]}
    for metric in ("cue_rate_mbps", "outage_rate", "mean_power_ratio", "v2v_delay_ms", "channels_used"):
        reference_row[LABELS[metric]] = ref[metric] * (100 if metric == "outage_rate" else 1)
    table.loc[f"{METHODS[method]['label']} (referencia)"] = pd.Series(reference_row)
table.round(2)

## Veredicto

Una linea por variante con los tres criterios, y como queda frente a la regla de umbral. La que los cumpla y mas se acerque a la regla es la candidata a configuracion de la tesis.

In [ ]:
passing = []
for name in variants:
    row = table.loc[name]
    collapses = row["Caida (%)"] < -100 * COLLAPSE_TOLERANCE
    learns = row["Final"] > row["Casi al azar"]
    beats = {m: row["Final"] > references[m]["reward"] for m in MUST_BEAT if m in references}
    checks = [
        "se cae al terminar la exploracion" if collapses else "no se cae",
        "termina por encima de su arranque" if learns else "termina por debajo de su arranque casi al azar",
    ]
    checks += [("le gana a " if ok else "no le gana a ") + METHODS[m]["label"] for m, ok in beats.items()]
    if "umbral" in references:
        gap = 100 * (row["Final"] - references["umbral"]["reward"]) / abs(references["umbral"]["reward"])
        checks.append(f"queda a {gap:+.0f} % de la regla de umbral")
    print(f"{name}: {', '.join(checks)} (caida {row['Caida (%)']:.1f} %).")
    if not collapses and learns and len(beats) == len(MUST_BEAT) and all(beats.values()):
        passing.append(name)
print()
if passing:
    print(f"Cumplen los tres criterios: {', '.join(passing)}.")
else:
    print("Ninguna variante cumple los tres criterios.")